In [2]:
# 1. Import Libraries
import numpy as np
import tensorflow as tf
from collections import deque
import time
import requests
import json
import os
import matplotlib.pyplot as plt 
from maze_environment import CustomMazeEnv

# Suppress TensorFlow warnings for cleaner console output
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# 2. Global Configuration & API Setup
# Note: It is highly recommended to mask this key or use environment variables before sharing/presenting
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

# Grid columns for coordinate transformation
GRID_COLS = 7

# 3. Environment Initialization
# Define the complex multi-objective conflict map (Goal+Prize+Trap)
conflict_map = [
    "SFFFFFG",
    "FTFPFTF",
    "FFFTFFF"
]

# Initialize global evaluation environment
global_env = CustomMazeEnv(expert_type='goal', custom_map=conflict_map)

# Extract state and action space dimensions for neural network configuration
n_states = global_env.observation_space.n
n_actions = global_env.action_space.n

print(f"Global environment initialized. State space: {n_states}, Action space: {n_actions}")

# Render the initial map to verify setup
print("\n--- Initial Conflict Map Preview ---")
global_env.reset()
global_env.render()

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 4. Neural Network Architecture
def build_dqn_model(n_states, n_actions):
    """
    Build and return a standard DQN model.
    Encapsulated as a function to allow easy reuse for instantiating 
    the three specialized experts and the global benchmark.
    """
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(1,)), 
        tf.keras.layers.Embedding(input_dim=n_states, output_dim=32),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n_actions)
    ])
    return model


# 5. DQN Core Components
def epsilon_greedy_policy(state, model, epsilon, n_actions):
    """Epsilon-Greedy policy to balance exploration and exploitation."""
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    else:
        # Note: Keras expects 2D array inputs
        Q_values = model.predict(np.array([[state]]), verbose=0)
        return np.argmax(Q_values[0])

def sample_experiences(replay_buffer, batch_size):
    """Randomly sample a batch from the experience replay buffer."""
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[index] for index in indices]
    return [
        np.array([experience[field_index] for experience in batch])
        for field_index in range(6)
    ] # Returns: states, actions, rewards, next_states, dones, truncateds

def training_step(batch_size, model, replay_buffer, optimizer, loss_fn, discount_factor, n_actions):
    """Execute a single gradient descent training step."""
    experiences = sample_experiences(replay_buffer, batch_size)
    states, actions, rewards, next_states, dones, truncateds = experiences

    # Calculate target Q-values
    next_Q_values = model.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    runs = 1.0 - (dones | truncateds)
    target_Q_values = rewards + runs * discount_factor * max_next_Q_values
    target_Q_values = target_Q_values.reshape(-1, 1)

    # Generate action mask
    mask = tf.one_hot(actions, n_actions)

    # Automatic differentiation and weight update
    with tf.GradientTape() as tape:
        all_Q_values = model(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    
    return loss

def play_one_step(env, state, model, replay_buffer, epsilon, n_actions):
    """Execute one step in the environment and store the experience in the replay buffer."""
    action = epsilon_greedy_policy(state, model, epsilon, n_actions)
    next_state, reward, done, truncated, info = env.step(action)
    replay_buffer.append((state, action, reward, next_state, done, truncated))
    return next_state, reward, done, truncated, info


# 6. Main Training Loop
def train_agent(env, model, num_iterations, optimizer, loss_fn, batch_size, discount_factor, epsilon_decay_steps=3000):
    """
    Refactored main training loop. 
    All dependencies are passed as arguments to prevent scope pollution.
    """
    replay_buffer = deque(maxlen=20000) # Expanded replay buffer for complex maps
    rewards_history = []
    losses_history = []
    
    print(f"Starting training... Total iterations: {num_iterations}")
    
    for episode in range(num_iterations):
        obs, info = env.reset()
        episode_reward = 0 
        
        for step in range(150): # Limit maximum steps to prevent infinite loops
            # Dynamically calculate Epsilon with time decay
            epsilon = max(1 - episode / epsilon_decay_steps, 0.10) 
            
            # Step and sample
            obs, reward, done, truncated, info = play_one_step(
                env, obs, model, replay_buffer, epsilon, env.action_space.n
            )
            episode_reward += reward

            # Start training once the replay buffer reaches the batch size
            if len(replay_buffer) > batch_size:
                loss = training_step(
                    batch_size, model, replay_buffer, optimizer, 
                    loss_fn, discount_factor, env.action_space.n
                )
                losses_history.append(float(loss))

            if done or truncated:
                break

        rewards_history.append(episode_reward)
        
        # Print progress (every 50 episodes)
        if (episode + 1) % 50 == 0:
            avg_reward = np.mean(rewards_history[-50:])
            print(f"Episode: {episode + 1}/{num_iterations} | "
                  f"Avg Reward (last 50): {avg_reward:.2f} | "
                  f"eps: {epsilon:.3f}")

    return model, rewards_history, losses_history

In [ ]:
# 7. Expert Initialization
import os

print("Initializing and loading pre-trained models...")

# 1. Instantiate the three specialized experts (input_dim=n_states, hidden units=32)
model_goal = build_dqn_model(n_states, n_actions)
model_prize = build_dqn_model(n_states, n_actions)
model_trap = build_dqn_model(n_states, n_actions)

# 2. Instantiate the expanded network for the Global Benchmark
# Note: The input dimension is doubled (n_states * 2 = 42) and Dense layers are increased to 64 units
model_global = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)), 
    tf.keras.layers.Embedding(input_dim=n_states * 2, output_dim=64),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(n_actions)
])

# 3. Define weight file paths
weights_paths = {
    "Goal Expert": ('model_goal', model_goal, 'goal_expert_open_map.weights.h5'),
    "Prize Expert": ('model_prize', model_prize, 'prize_expert_open_map.weights.h5'),
    "Trap Expert": ('model_trap', model_trap, 'trap_expert_open_map.weights.h5'),
    "Global Benchmark": ('model_global', model_global, 'global_benchmark_open_map.weights.h5')
}

# 4. Load weights if available
models_loaded_successfully = True

for model_name, (var_name, model_instance, filepath) in weights_paths.items():
    if os.path.exists(filepath):
        model_instance.load_weights(filepath)
        print(f"Loaded {model_name} weights successfully ({filepath})")
    else:
        print(f"Warning: Weight file for {model_name} not found ({filepath})")
        models_loaded_successfully = False

if not models_loaded_successfully:
    print("\nNote: Please train all models and ensure their .weights.h5 files are in the working directory.")
else:
    print("\nAll expert and benchmark models are ready for evaluation.")

In [ ]:
# 8. Routing Strategies (Routers)
import time
import requests
import json
import numpy as np

print("Initializing gating routing strategies...")

# ------------------------------------------
# Strategy 1: Random Baseline
# ------------------------------------------
def random_gating_router():
    """
    Randomly selects an expert at each step.
    Serves as the performance lower bound baseline.
    """
    return np.random.choice(['trap', 'prize', 'goal'])


# ------------------------------------------
# Strategy 2: Smart Heuristic
# ------------------------------------------
def heuristic_gating_router(state, prize_collected):
    """
    Heuristic gating function based on human prior knowledge.
    Prioritizes the 'trap' expert if adjacent to scattered traps.
    Otherwise, selects 'prize' or 'goal' based on collection status.
    """
    row = state // 7
    col = state % 7
    
    # Exact trap coordinates in the custom map
    traps = [(1, 1), (1, 5), (2, 3)]
    
    # Check for adjacency to any trap (Manhattan distance == 1)
    is_near_trap = any(abs(row - tr) + abs(col - tc) == 1 for tr, tc in traps)
    
    if is_near_trap:
        return 'trap'
    elif not prize_collected:
        return 'prize'
    else:
        return 'goal'


# ------------------------------------------
# Strategy 3: LLM-enabled MoE Router (Gemini)
# ------------------------------------------
# API Key placeholder (Ensure to mask before sharing/presenting)
API_KEY = GEMINI_API_KEY

def llm_gating_router_safe(state, prize_collected, max_retries=5):
    """
    LLM-enabled MoE routing using Gemini API.
    Includes exponential backoff for rate limits and a heuristic fallback mechanism.
    """
    row = state // 7
    col = state % 7
    
    # Tailored prompt for the complex grid environment
    prompt = f"""
    You are the central routing intelligence for an AI agent in a 3x7 grid maze.
    
    Map Layout:
    - Start point (S) is at (0, 0).
    - Goal (G) is at (0, 6).
    - Prize (P) is at (1, 3).
    - Traps (T) are scattered at coordinates (1, 1), (1, 5), and (2, 3).
    - All other grids are safe floors.
    
    Current Status:
    - The Agent is currently at Row {row}, Column {col}.
    - Has the prize been collected? {prize_collected}.
    
    You have three experts to choose from to make the next move:
    1. "prize": Navigates directly towards the Prize.
    2. "goal": Navigates directly towards the Goal.
    3. "trap": Focuses on safely stepping away from traps.
    
    Your Mission:
    - You must collect the prize before reaching the goal.
    - If you are adjacent to a Trap, you might need to prioritize the "trap" expert to avoid stepping on it.
    - Otherwise, select "prize" if the prize is not yet collected, or "goal" if it is.
    
    CRITICAL INSTRUCTION: You must output EXACTLY ONE WORD from the following list: [prize, goal, trap]. Do not output any punctuation, explanations, or extra spaces.
    """
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    headers = {'Content-Type': 'application/json'}
    data = {
        "contents": [{"parts": [{"text": prompt}]}]
    }
    
    # Automatic retry mechanism for 503/429 HTTP errors
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=data)
            
            if response.status_code == 200:
                response_data = response.json()
                text_output = response_data['candidates'][0]['content']['parts'][0]['text']
                decision = text_output.strip().lower()
                
                # Filter out potential punctuation (e.g., if LLM outputs "goal.")
                decision = decision.replace(".", "").replace("\"", "").replace("\'", "")
                
                if decision in ['prize', 'goal', 'trap']:
                    return decision
                else:
                    print(f"  [Parsing Warning] Non-standard output from LLM: '{decision}'. Triggering heuristic fallback.")
                    return heuristic_gating_router(state, prize_collected)
                    
            elif response.status_code in [503, 500, 429]:
                wait_time = (2 ** attempt) + np.random.uniform(0, 1)
                print(f"  [API Busy] Status {response.status_code}, retrying in {wait_time:.1f}s (Attempt {attempt+1})...")
                time.sleep(wait_time)
                continue
                
            else:
                print(f"  [API Error] Status code: {response.status_code}. Triggering heuristic fallback.")
                return heuristic_gating_router(state, prize_collected)
                
        except Exception as e:
            print(f"  [Network Exception] {e}, retrying...")
            time.sleep(2)
            
    print("  [Retries Exhausted] Falling back to heuristic router.")
    return heuristic_gating_router(state, prize_collected)

print("LLM-enabled robust router is ready.")

In [ ]:
# 8. Routing Strategies (Routers)
import time
import requests
import json
import numpy as np

print("Initializing gating routing strategies...")

# ------------------------------------------
# Strategy 1: Random Baseline
# ------------------------------------------
def random_gating_router():
    """
    Randomly selects an expert at each step.
    Serves as the performance lower bound baseline.
    """
    return np.random.choice(['trap', 'prize', 'goal'])


# ------------------------------------------
# Strategy 2: Smart Heuristic
# ------------------------------------------
def heuristic_gating_router(state, prize_collected):
    """
    Heuristic gating function based on human prior knowledge.
    Prioritizes the 'trap' expert if adjacent to scattered traps.
    Otherwise, selects 'prize' or 'goal' based on collection status.
    """
    row = state // 7
    col = state % 7
    
    # Exact trap coordinates in the custom map
    traps = [(1, 1), (1, 5), (2, 3)]
    
    # Check for adjacency to any trap (Manhattan distance == 1)
    is_near_trap = any(abs(row - tr) + abs(col - tc) == 1 for tr, tc in traps)
    
    if is_near_trap:
        return 'trap'
    elif not prize_collected:
        return 'prize'
    else:
        return 'goal'


# ------------------------------------------
# Strategy 3: LLM-enabled MoE Router (Gemini)
# ------------------------------------------
# API Key placeholder (Ensure to mask before sharing/presenting)
API_KEY = GEMINI_API_KEY

def llm_gating_router_safe(state, prize_collected, max_retries=5):
    """
    LLM-enabled MoE routing using Gemini API.
    Includes exponential backoff for rate limits and a heuristic fallback mechanism.
    """
    row = state // 7
    col = state % 7
    
    # Tailored prompt for the complex grid environment
    prompt = f"""
    You are the central routing intelligence for an AI agent in a 3x7 grid maze.
    
    Map Layout:
    - Start point (S) is at (0, 0).
    - Goal (G) is at (0, 6).
    - Prize (P) is at (1, 3).
    - Traps (T) are scattered at coordinates (1, 1), (1, 5), and (2, 3).
    - All other grids are safe floors.
    
    Current Status:
    - The Agent is currently at Row {row}, Column {col}.
    - Has the prize been collected? {prize_collected}.
    
    You have three experts to choose from to make the next move:
    1. "prize": Navigates directly towards the Prize.
    2. "goal": Navigates directly towards the Goal.
    3. "trap": Focuses on safely stepping away from traps.
    
    Your Mission:
    - You must collect the prize before reaching the goal.
    - If you are adjacent to a Trap, you might need to prioritize the "trap" expert to avoid stepping on it.
    - Otherwise, select "prize" if the prize is not yet collected, or "goal" if it is.
    
    CRITICAL INSTRUCTION: You must output EXACTLY ONE WORD from the following list: [prize, goal, trap]. Do not output any punctuation, explanations, or extra spaces.
    """
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    headers = {'Content-Type': 'application/json'}
    data = {
        "contents": [{"parts": [{"text": prompt}]}]
    }
    
    # Automatic retry mechanism for 503/429 HTTP errors
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=data)
            
            if response.status_code == 200:
                response_data = response.json()
                text_output = response_data['candidates'][0]['content']['parts'][0]['text']
                decision = text_output.strip().lower()
                
                # Filter out potential punctuation (e.g., if LLM outputs "goal.")
                decision = decision.replace(".", "").replace("\"", "").replace("\'", "")
                
                if decision in ['prize', 'goal', 'trap']:
                    return decision
                else:
                    print(f"  [Parsing Warning] Non-standard output from LLM: '{decision}'. Triggering heuristic fallback.")
                    return heuristic_gating_router(state, prize_collected)
                    
            elif response.status_code in [503, 500, 429]:
                wait_time = (2 ** attempt) + np.random.uniform(0, 1)
                print(f"  [API Busy] Status {response.status_code}, retrying in {wait_time:.1f}s (Attempt {attempt+1})...")
                time.sleep(wait_time)
                continue
                
            else:
                print(f"  [API Error] Status code: {response.status_code}. Triggering heuristic fallback.")
                return heuristic_gating_router(state, prize_collected)
                
        except Exception as e:
            print(f"  [Network Exception] {e}, retrying...")
            time.sleep(2)
            
    print("  [Retries Exhausted] Falling back to heuristic router.")
    return heuristic_gating_router(state, prize_collected)

print("LLM-enabled robust router is ready.")

In [ ]:
# 8. With Latency detection
import time
import requests
import json
import numpy as np

def benchmark_router_latency(num_trials=20):
    print(f"--- Benchmarking Router Latency (Trials: {num_trials}) ---")
    
    # Simulate a fixed intermediate state
    test_state = 10 
    test_prize_collected = False
    
    # 1. Measure the heuristic router (a local conventional gating network)
    heuristic_latencies = []
    for _ in range(num_trials):
        start_time = time.perf_counter()
        _ = heuristic_gating_router(test_state, test_prize_collected)
        end_time = time.perf_counter()
        heuristic_latencies.append((end_time - start_time) * 1000) # Convert to milliseconds (ms)
        
    print(f"Heuristic Router : {np.mean(heuristic_latencies):.4f} ms ± {np.std(heuristic_latencies):.4f} ms")
    
    # 2. Measure the LLM router (Gemini API call)
    llm_latencies = []
    print("Testing LLM latency (this will take some time due to API calls)...")
    for i in range(num_trials):
        start_time = time.perf_counter()
        _ = llm_gating_router_safe(test_state, test_prize_collected)
        end_time = time.perf_counter()
        llm_latencies.append((end_time - start_time) * 1000) # Convert to milliseconds (ms)
        print(f"  - Trial {i+1}: {llm_latencies[-1]:.2f} ms")
        time.sleep(1) # Avoid triggering the API rate limit
        
    print(f"LLM (Gemini) Router: {np.mean(llm_latencies):.2f} ms ± {np.std(llm_latencies):.2f} ms")

# Run the performance test
benchmark_router_latency(num_trials=20)
print("Initializing gating routing strategies...")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Use the measured latency data
routers = ['Smart Heuristic\n(Local DNN Gate)', 'LLM-Enabled\n(Gemini 2.5 Flash API)']
mean_latencies = [0.0009, 9669.10]  # Measured mean latency (ms)
std_latencies = [0.0009, 7790.49]   # Measured standard deviation (ms)

# Configure the plot style
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(6, 5))

# Draw the bar chart
bars = ax.bar(routers, mean_latencies, yerr=std_latencies, capsize=8, 
              color=['#2ca02c', '#d62728'], alpha=0.8, edgecolor='black', width=0.5)

# Use a logarithmic scale for the y-axis
ax.set_yscale('log')

# Set the title and labels
ax.set_ylabel('Inference Latency (ms) [Log Scale]', fontsize=12, fontweight='bold')
ax.set_title('Inference Latency Comparison', fontsize=14, fontweight='bold')

# Set a high y-axis limit (for example, 100000) so the 17000+ ms error bar fits
ax.set_ylim(0.0001, 100000) 

# Annotate the bars using suitable decimal precision
for i, bar in enumerate(bars):
    height = bar.get_height()
    # Use four decimal places for the local algorithm and two for the LLM
    if i == 0:
        label = f'{mean_latencies[i]:.4f} ms\n±{std_latencies[i]:.4f}'
    else:
        label = f'{mean_latencies[i]:.2f} ms\n±{std_latencies[i]:.2f}'
        
    ax.text(bar.get_x() + bar.get_width() / 2, height * 1.5, 
            label, 
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
# 9. Unified Evaluation Loop
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # Used for generating advanced statistical plots (e.g., heatmaps, box plots)

def evaluate_strategy(env, strategy_name, models_dict, num_episodes=50, max_steps=100):
    """
    Executes N evaluation episodes and collects statistical metrics for a given strategy.
    """
    print(f"\nStarting evaluation for strategy: [{strategy_name.upper()}] ({num_episodes} episodes)...")
    
    stats = {
        'success_count': 0,
        'prize_collected_count': 0,
        'trap_hit_count': 0,
        'steps_taken': [],      # Records step counts only for successful episodes (reached Goal)
        'total_rewards': [],    # Records cumulative rewards across all episodes
        'expert_usage': {'goal': 0, 'prize': 0, 'trap': 0} # Tracks expert invocation frequency
    }
    
    for episode in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        
        # Print progress to indicate active execution
        if (episode + 1) % 10 == 0:
            print(f"  > Progress: {episode + 1}/{num_episodes} episodes completed")
            
        for step in range(max_steps):
            is_prize_collected = env.prize_collected
            obs_input = np.array([[obs]], dtype=np.int32)
            
            # 1. Determine which expert takes control
            if strategy_name == 'global':
                # Important: Simulate the state-shifting logic of the POMDP wrapper for the global benchmark
                global_obs = obs + (21 if is_prize_collected else 0)
                global_obs_input = np.array([[global_obs]], dtype=np.int32)
                q_values = models_dict['global'].predict(global_obs_input, verbose=0)
            else:
                # Utilize the specified router to select the active expert
                if strategy_name == 'random':
                    selected_expert = random_gating_router()
                elif strategy_name == 'heuristic':
                    selected_expert = heuristic_gating_router(obs, is_prize_collected)
                elif strategy_name == 'llm':
                    selected_expert = llm_gating_router_safe(obs, is_prize_collected)
                
                stats['expert_usage'][selected_expert] += 1
                q_values = models_dict[selected_expert].predict(obs_input, verbose=0)
            
            # 2. Execute Action
            action = np.argmax(q_values[0])
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            
            if terminated or truncated or step == max_steps - 1:
                cell_type = info.get('cell_type')
                if cell_type == 'G':
                    stats['success_count'] += 1
                    stats['steps_taken'].append(step + 1)
                elif cell_type == 'T':
                    stats['trap_hit_count'] += 1
                
                # Count the prize as collected even if the episode fails or times out
                if env.prize_collected:
                    stats['prize_collected_count'] += 1
                    
                break
                
        stats['total_rewards'].append(episode_reward)
        
    print(f"Evaluation for {strategy_name.upper()} strategy completed.")
    return stats

In [ ]:
# 10. Execute Comparative Experiments (with Local Caching)
import os
import pickle

# Package models into a dictionary for easier reference during evaluation
all_models = {
    'goal': model_goal,
    'prize': model_prize,
    'trap': model_trap,
    'global': model_global
}

# Evaluation configuration
# Note: 50 or 100 episodes are recommended for stable statistical results
NUM_EVAL_EPISODES = 50
RESULTS_FILE = 'moe_evaluation_results.pkl' # Cache file name
FORCE_RERUN = False  # Set to True to bypass the cache and re-run the evaluation

# Initialize the environment for evaluation using the defined conflict map
global_env = CustomMazeEnv(expert_type='global', custom_map=conflict_map)

# Implement local caching to avoid redundant, time-consuming LLM API calls
if os.path.exists(RESULTS_FILE) and not FORCE_RERUN:
    print(f"Local cache '{RESULTS_FILE}' found. Loading data...")
    with open(RESULTS_FILE, 'rb') as f:
        all_results = pickle.load(f)
    print("Cached data loaded successfully. Ready for visualization.")
else:
    print("No cache found or FORCE_RERUN triggered. Starting evaluation...")
    
    # Dictionary to store evaluation metrics for all strategies
    all_results = {}

    # 1. Evaluate Random Router
    all_results['Random'] = evaluate_strategy(global_env, 'random', all_models, NUM_EVAL_EPISODES)

    # 2. Evaluate Smart Heuristic Router
    all_results['Heuristic'] = evaluate_strategy(global_env, 'heuristic', all_models, NUM_EVAL_EPISODES)

    # 3. Evaluate LLM Router (Gemini API - time-consuming)
    all_results['LLM (Gemini)'] = evaluate_strategy(global_env, 'llm', all_models, NUM_EVAL_EPISODES)

    # 4. Evaluate Global Benchmark
    all_results['Global Benchmark'] = evaluate_strategy(global_env, 'global', all_models, NUM_EVAL_EPISODES)

    # Save results to local cache
    print(f"\nEvaluation complete. Saving data to '{RESULTS_FILE}'...")
    with open(RESULTS_FILE, 'wb') as f:
        pickle.dump(all_results, f)

    print("All strategy evaluation data collected and saved.")

In [ ]:
# 11. Academic-Grade Data Visualization
# Extract plotting data from the evaluation results
strategies = list(all_results.keys())
success_rates = [all_results[s]['success_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]
prize_rates = [all_results[s]['prize_collected_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]
trap_rates = [all_results[s]['trap_hit_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]

# Set global plotting style
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(16, 6))

# --- Plot 1: Multi-Objective Completion Rates (Grouped Bar Chart) ---
ax1 = plt.subplot(1, 2, 1)
x = np.arange(len(strategies))
width = 0.25

rects1 = ax1.bar(x - width, success_rates, width, label='Reached Goal (Success)', color='#2ca02c')
rects2 = ax1.bar(x, prize_rates, width, label='Collected Prize', color='#ff7f0e')
rects3 = ax1.bar(x + width, trap_rates, width, label='Hit Trap (Failure)', color='#d62728')

ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_title('Multi-Objective Task Completion Rates by Strategy', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(strategies, fontsize=11)
ax1.legend()
ax1.set_ylim(0, 110)

# Function to add data labels above bars
def autolabel(rects):
    """Attach a text label above each bar displaying its height."""
    for rect in rects:
        height = rect.get_height()
        if height > 0:
            ax1.annotate(f'{height:.0f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

# --- Plot 2: Efficiency Distribution for Successful Episodes (Boxplot) ---
ax2 = plt.subplot(1, 2, 2)
steps_data = [all_results[s]['steps_taken'] for s in strategies]

# Generate boxplot using Seaborn
sns.boxplot(data=steps_data, palette="Set2", ax=ax2, width=0.5)
ax2.set_xticklabels(strategies, fontsize=11)
ax2.set_ylabel('Steps Taken (Lower is better)', fontsize=12)
ax2.set_title('Efficiency Distribution (Successful Episodes Only)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Output average cumulative rewards for reference
print("\n=== Average Cumulative Rewards by Strategy ===")
for s in strategies:
    avg_reward = np.mean(all_results[s]['total_rewards'])
    print(f"{s}: {avg_reward:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 11. Academic-Grade Data Visualization (Enhanced with Statistical Significance)
# Extract plotting data from the evaluation results
strategies = list(all_results.keys())
success_rates = [all_results[s]['success_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]
prize_rates = [all_results[s]['prize_collected_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]
trap_rates = [all_results[s]['trap_hit_count'] / NUM_EVAL_EPISODES * 100 for s in strategies]

# Extract steps and rewards arrays for variance calculations
steps_data = [all_results[s]['steps_taken'] for s in strategies]
rewards_data = [all_results[s]['total_rewards'] for s in strategies]

# Calculate Means and Standard Deviations for Rewards
reward_means = [np.mean(r) for r in rewards_data]
reward_stds = [np.std(r) for r in rewards_data]

# Set global plotting style
sns.set_theme(style="whitegrid")
# Expanded figure size to accommodate a 3rd subplot
fig = plt.figure(figsize=(20, 6)) 

# --- Plot 1: Multi-Objective Completion Rates (Grouped Bar Chart) ---
ax1 = plt.subplot(1, 3, 1)
x = np.arange(len(strategies))
width = 0.25

rects1 = ax1.bar(x - width, success_rates, width, label='Reached Goal (Success)', color='#2ca02c')
rects2 = ax1.bar(x, prize_rates, width, label='Collected Prize', color='#ff7f0e')
rects3 = ax1.bar(x + width, trap_rates, width, label='Hit Trap (Failure)', color='#d62728')

ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_title('Task Completion Rates', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(strategies, fontsize=11)
ax1.legend()
ax1.set_ylim(0, 110)

# Function to add data labels above bars
def autolabel(rects, ax):
    """Attach a text label above each bar displaying its height."""
    for rect in rects:
        height = rect.get_height()
        if height > 0:
            ax.annotate(f'{height:.0f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

autolabel(rects1, ax1)
autolabel(rects2, ax1)
autolabel(rects3, ax1)

# --- Plot 2: Efficiency Distribution (Boxplot + Stripplot) ---
ax2 = plt.subplot(1, 3, 2)
# Draw the boxplot (hide outliers as stripplot will show them)
sns.boxplot(data=steps_data, palette="Set2", ax=ax2, width=0.5, showfliers=False)
# Overlay a stripplot to show individual data points
# This visually proves "Zero-Variance" for LLM/Heuristic as all points will perfectly overlap!
sns.stripplot(data=steps_data, color=".25", ax=ax2, alpha=0.6, jitter=True, size=5)

ax2.set_xticklabels(strategies, fontsize=11)
ax2.set_ylabel('Steps Taken (Lower is better)', fontsize=12)
ax2.set_title('Efficiency (Successful Episodes)', fontsize=14, fontweight='bold')

# --- Plot 3: Average Cumulative Rewards with Error Bars ---
ax3 = plt.subplot(1, 3, 3)
# Draw bar chart with yerr parameter for Standard Deviation
ax3.bar(x, reward_means, yerr=reward_stds, capsize=8, color='#4c6ef5', alpha=0.8, edgecolor='black')

ax3.set_ylabel('Cumulative Reward', fontsize=12)
ax3.set_title('Average Rewards with Standard Deviation', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(strategies, fontsize=11)

# Annotate Mean and SD inside the bars for quick reading
for i in range(len(strategies)):
    if reward_means[i] > 0:
        ax3.text(x[i], reward_means[i] * 0.5, f'{reward_means[i]:.1f}\n±{reward_stds[i]:.1f}', 
                 ha='center', va='center', color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# ========================================================
# Output formatted statistical data for Paper/Table inclusion
# ========================================================
print("\n=== Statistical Summary (Mean ± Standard Deviation) ===")
for i, s in enumerate(strategies):
    # Calculate step metrics explicitly for text output
    valid_steps = steps_data[i]
    if len(valid_steps) > 0:
        steps_mean = np.mean(valid_steps)
        steps_std = np.std(valid_steps)
    else:
        steps_mean, steps_std = 0, 0
        
    print(f"Strategy: [{s.upper()}]")
    print(f"  - Average Reward : {reward_means[i]:.2f} ± {reward_stds[i]:.2f}")
    print(f"  - Steps Taken    : {steps_mean:.2f} ± {steps_std:.2f}")
    print("-" * 45)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import tensorflow as tf # Ensure tf is imported for tensor conversion

def visualize_trajectory_timeline(env, models_dict, strategy_name='llm'):
    """
    Generate an academic-grade trajectory footprint timeline.
    Adapted for the complex open map (Goal+Prize+Trap).
    """
    obs, info = env.reset()
    
    # Record trajectory coordinates and the corresponding routing expert
    path_coords = [(obs // 7, obs % 7)]
    expert_choices = ['start']
    
    print(f"Generating footprint trajectory for strategy: [{strategy_name.upper()}]...")
    
    # 1. Run evaluation and collect trajectory data
    for step in range(50):
        is_prize_collected = env.prize_collected
        
        # Optimization: Use tf.convert_to_tensor to avoid Keras shape warnings
        if strategy_name == 'global':
            global_obs = obs + (21 if is_prize_collected else 0)
            obs_tensor = tf.convert_to_tensor([[global_obs]], dtype=tf.int32)
            q_values = models_dict['global'](obs_tensor, training=False).numpy()
            expert_choices.append('global')
        else:
            obs_tensor = tf.convert_to_tensor([[obs]], dtype=tf.int32)
            if strategy_name == 'llm':
                expert = llm_gating_router_safe(obs, is_prize_collected)
            elif strategy_name == 'heuristic':
                expert = heuristic_gating_router(obs, is_prize_collected) 
            else:
                expert = random_gating_router()
                
            q_values = models_dict[expert](obs_tensor, training=False).numpy()
            expert_choices.append(expert)
                
        action = np.argmax(q_values[0])
        obs, reward, terminated, truncated, info = env.step(action)
        
        row, col = obs // 7, obs % 7
        path_coords.append((row, col))
        
        if terminated or truncated:
            break

    # 2. Visualize the grid maze
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(2.5, -0.5) # Invert Y-axis so (0,0) is at the top-left
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Define landmark coordinates for the custom map
    traps = [(1, 1), (1, 5), (2, 3)]
    prize = (1, 3)
    start = (0, 0)
    goal = (0, 6)
    
    # Draw grid cells and background colors
    for r in range(3):
        for c in range(7):
            current_pos = (r, c)
            color = '#f8f9fa' # Safe floor (Light gray)
            
            if current_pos in traps: color = '#ffe3e3'     # Trap (Light red)
            elif current_pos == start: color = '#e7f5ff'   # Start (Light blue)
            elif current_pos == goal: color = '#ebfbee'    # Goal (Light green)
            elif current_pos == prize: color = '#fff3bf'   # Prize (Light yellow)
            
            rect = Rectangle((c-0.5, r-0.5), 1, 1, facecolor=color, edgecolor='#dee2e6', linewidth=2)
            ax.add_patch(rect)
            
            # Annotate landmark text
            if current_pos in traps: 
                ax.text(c, r, 'TRAP', ha='center', va='center', color='#fa5252', weight='bold', alpha=0.3, fontsize=14)
            elif current_pos == start: 
                ax.text(c, r, 'START', ha='center', va='center', color='#228be6', weight='bold', alpha=0.5, fontsize=14)
            elif current_pos == goal: 
                ax.text(c, r, 'GOAL', ha='center', va='center', color='#40c057', weight='bold', alpha=0.5, fontsize=14)
            elif current_pos == prize: 
                ax.text(c, r, 'PRIZE', ha='center', va='center', color='#fcc419', weight='bold', alpha=0.5, fontsize=14)

    # 3. Draw trajectory footprint connections and step numbers
    # Track cell visits to apply slight coordinate offsets, preventing overlapping lines during revisits
    cell_visits = {}
    
    for i in range(len(path_coords)):
        r, c = path_coords[i]
        
        # Calculate offset (shift slightly down-right for each revisit to the same cell)
        visits = cell_visits.get((r, c), 0)
        offset_x = visits * 0.15 - 0.1
        offset_y = visits * 0.15 - 0.1
        cell_visits[(r, c)] = visits + 1
        
        current_x = c + offset_x
        current_y = r + offset_y
        
        # Plot footprint nodes and step numbers
        ax.plot(current_x, current_y, marker='o', markersize=18, color='#4c6ef5', alpha=0.9, zorder=5)
        ax.text(current_x, current_y, str(i), color='white', ha='center', va='center', fontsize=10, weight='bold', zorder=6)
        
        # Draw connecting lines
        if i > 0:
            prev_r, prev_c = path_coords[i-1]
            prev_visits = cell_visits[(prev_r, prev_c)] - 1
            prev_x = prev_c + (prev_visits * 0.15 - 0.1)
            prev_y = prev_r + (prev_visits * 0.15 - 0.1)
            
            # Connect consecutive footprints with arrows
            ax.annotate("", xy=(current_x, current_y), xytext=(prev_x, prev_y),
                        arrowprops=dict(arrowstyle="-|>", color="#4c6ef5", lw=2, shrinkA=10, shrinkB=10), zorder=4)

    ax.set_title(f'Agent Trajectory Footprints (Strategy: {strategy_name.upper()})', fontsize=16, pad=15, weight='bold')
    plt.tight_layout()
    plt.show()

# Execute visualization
visualize_trajectory_timeline(global_env, all_models, strategy_name='llm')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import random

def simulate_random_agent_clean(max_steps=50):
    """
    Simulate a random policy and omit repeated positions caused by walking into walls.
    """
    traps = [(1, 1), (1, 5), (2, 3)]
    goal = (0, 6)
    start = (0, 0)
    
    pos = start
    path_coords = [pos]
    actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    actual_steps_taken = 0 
    
    for _ in range(max_steps):
        actual_steps_taken += 1
        move = random.choice(actions)
        new_r = pos[0] + move[0]
        new_c = pos[1] + move[1]
        
        if new_r < 0 or new_r > 2 or new_c < 0 or new_c > 6:
            new_r, new_c = pos[0], pos[1]
            
        new_pos = (new_r, new_c)
        
        # Record only transitions that change the physical position
        if new_pos != pos:
            path_coords.append(new_pos)
            
        pos = new_pos
        
        if pos in traps:
            return path_coords, 'trap', actual_steps_taken
        if pos == goal:
            return path_coords, 'goal', actual_steps_taken
            
    return path_coords, 'timeout', actual_steps_taken

def capture_real_detour_trajectory_clean():
    print("Hunting for a clean detour trajectory...")
    while True:
        path, outcome, total_steps = simulate_random_agent_clean(max_steps=50)
        if outcome == 'goal' and 10 < len(path) < 20:
            print(f"Found! Effective moves: {len(path)-1}. Total timesteps (incl. wall bumps): {total_steps}")
            return path

# ==========================================
# Plot with vertical alignment and straight connecting lines
# ==========================================
def visualize_real_trajectory_vertical(path_coords, strategy_name='Random Baseline', outcome='Inefficient Detour'):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(2.5, -0.5) 
    ax.set_xticks([])
    ax.set_yticks([])
    
    traps = [(1, 1), (1, 5), (2, 3)]
    prize = (1, 3)
    start = (0, 0)
    goal = (0, 6)
    
    # 1. Draw the grid background
    for r in range(3):
        for c in range(7):
            current_pos = (r, c)
            color = '#f8f9fa' 
            if current_pos in traps: color = '#ffe3e3'
            elif current_pos == start: color = '#e7f5ff'
            elif current_pos == goal: color = '#ebfbee'
            elif current_pos == prize: color = '#fff3bf'
            
            rect = Rectangle((c-0.5, r-0.5), 1, 1, facecolor=color, edgecolor='#dee2e6', linewidth=2)
            ax.add_patch(rect)
            
            if current_pos in traps: 
                ax.text(c, r, 'TRAP', ha='center', va='center', color='#fa5252', weight='bold', alpha=0.3, fontsize=14)
            elif current_pos == start: 
                ax.text(c, r, 'START', ha='center', va='center', color='#228be6', weight='bold', alpha=0.5, fontsize=14)
            elif current_pos == goal: 
                ax.text(c, r, 'GOAL', ha='center', va='center', color='#40c057', weight='bold', alpha=0.5, fontsize=14)
            elif current_pos == prize: 
                ax.text(c, r, 'PRIZE', ha='center', va='center', color='#fcc419', weight='bold', alpha=0.5, fontsize=14)

    # 2. Count visits to each cell for vertically centered placement
    total_visits_map = {}
    for p in path_coords:
        total_visits_map[p] = total_visits_map.get(p, 0) + 1

    # Generate all plotting coordinates
    plot_coords = []
    cell_visit_counter = {}
    
    for r, c in path_coords:
        visits = cell_visit_counter.get((r, c), 0)
        total = total_visits_map[(r, c)]
        
        offset_x = 0  # Keep all points horizontally aligned on the cell centerline
        
        # Distribute visits vertically across the [-0.3, 0.3] range within each cell
        if total == 1:
            offset_y = 0
        else:
            # Limit the maximum spacing so points remain inside the cell
            spacing = min(0.25, 0.6 / (total - 1))
            start_y = - ((total - 1) * spacing) / 2
            offset_y = start_y + visits * spacing
            
        cell_visit_counter[(r, c)] = visits + 1
        plot_coords.append((c + offset_x, r + offset_y)) # The plotting coordinates (x, y) correspond to (column, row)

    # 3. Draw nodes and straight connecting lines
    path_color = '#fd7e14' 
    
    for i in range(len(plot_coords)):
        current_x, current_y = plot_coords[i]
        
        # Draw the nodes
        ax.plot(current_x, current_y, marker='o', markersize=18, color=path_color, alpha=0.9, zorder=5)
        ax.text(current_x, current_y, str(i), color='white', ha='center', va='center', fontsize=10, weight='bold', zorder=6)
        
        # Draw straight connecting lines (without curved edges)
        if i > 0:
            prev_x, prev_y = plot_coords[i-1]
            ax.annotate("", xy=(current_x, current_y), xytext=(prev_x, prev_y),
                        arrowprops=dict(arrowstyle="-|>", color=path_color, lw=2, shrinkA=10, shrinkB=10), zorder=4)

    ax.set_title(f'Agent Trajectory Footprints\n(Strategy: {strategy_name.upper()} | Outcome: {outcome})', 
                 fontsize=16, pad=15, weight='bold')
    plt.tight_layout()
    plt.show()

# Run the visualization
clean_path = capture_real_detour_trajectory_clean()
visualize_real_trajectory_vertical(clean_path, outcome='Inefficient Detour (Missed Prize)')